In [1]:
# Colab: 挂载 Google Drive + Notebook 自备份
import os
from pathlib import Path

if os.path.exists("/content"):
    from google.colab import drive

    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    else:
        print("Drive already mounted.")

    # ========== Notebook 自备份到 Drive ==========
    # 每次在 Colab 中运行时，自动把当前 notebook 保存到 Drive
    # 这样只需手动上传 1 次，后续改动会在运行时自动覆盖 Drive 版本
    try:
        from google.colab import _message
        import json as _json

        resp = _message.blocking_request('get_ipynb', request='', timeout_sec=5)
        if resp and 'ipynb' in resp:
            # 探测 Drive 上的目标路径
            _drive_candidates = [
                Path("/content/drive/MyDrive/ML_project/notebooks"),
                Path("/content/drive/MyDrive/ML_project"),
                Path("/content/drive/MyDrive/course project/notebooks"),
                Path("/content/drive/MyDrive/course project"),
                Path("/content/drive/MyDrive/新建文件夹/course project/notebooks"),
                Path("/content/drive/MyDrive/新建文件夹/course project"),
            ]
            for _p in _drive_candidates:
                if _p.exists():
                    _dst = _p / "train_colab.ipynb"
                    with open(_dst, 'w', encoding='utf-8') as _f:
                        _json.dump(resp['ipynb'], _f, indent=1, ensure_ascii=False)
                    print(f"  ✅ Notebook 已自备份到: {_dst}")
                    break
            else:
                print("  ℹ️ 未找到 Drive 目标路径，跳过自备份")
    except Exception as _e:
        print(f"  ℹ️ Notebook 自备份跳过: {_e}")
else:
    print("Non-Colab environment, skip Drive mount.")


Mounted at /content/drive


## 配置：数据集与模型选择
在这里设定要跑哪些数据集、哪些模型，以及训练参数。

In [2]:
# ================================================================
# ★ 实验配置 — 根据需要修改这里 ★
# ================================================================

# 数据集列表：可选 'BCIC2A', 'CHINESE', 'MDD', 'SEED', 'SLEEP'
DATASETS = ["BCIC2A", "CHINESE", "MDD", "SEED", "SLEEP"]

# 模型列表：可选 'SimpleLinear', 'SimpleMLP', 'EEGNet', 'EEGGRU'
MODELS = ["SimpleLinear", "SimpleMLP", "EEGNet", "EEGGRU"]

# 训练参数
EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-4

# 是否跳过已存在的预测文件（断点续跑）
SKIP_EXISTING = True

# Colab 下是否复制到本地缓存加速
USE_LOCAL_CACHE = True

# 手动指定数据根目录（None 或 "" 则自动探测）
# Colab：把整个 ML_project 同步到 Drive 后，通常路径为：
# DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/ML_project/data/course project"
DATA_ROOT_OVERRIDE = None

# 是否将结果回写到 Google Drive（仅在 Colab 且 USE_LOCAL_CACHE 时生效）
SYNC_TO_DRIVE = True

print("=== 实验配置 ===")
print(f"数据集: {DATASETS}")
print(f"模型:   {MODELS}")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}")
print(f"回写 Drive: {SYNC_TO_DRIVE}")
print(f"共 {len(DATASETS)} 个数据集 × {len(MODELS)} 个模型 = {len(DATASETS) * len(MODELS)} 组实验")


=== 实验配置 ===
数据集: ['BCIC2A', 'CHINESE', 'MDD', 'SEED', 'SLEEP']
模型:   ['SimpleLinear', 'SimpleMLP', 'EEGNet', 'EEGGRU']
Epochs: 30, Batch: 32, LR: 0.0001
回写 Drive: True
共 5 个数据集 × 4 个模型 = 20 组实验


## 数据路径自动探测
自动定位 `course project` 数据目录，支持 Colab Drive 和本地运行。

In [3]:
import json
import os
from pathlib import Path

_DATASET_MARKERS = ("BCIC2A", "CHINESE", "MDD", "SEED", "SLEEP")


def _is_valid_data_root(p: Path) -> bool:
    """Root must contain at least one dataset folder with train.h5."""
    if not p.is_dir():
        return False
    return any((p / name / "train.h5").exists() for name in _DATASET_MARKERS)


def _data_root_candidates():
    drive = Path("/content/drive/MyDrive")
    candidates = [
        Path.cwd() / "data" / "course project",
        Path.cwd().parent / "data" / "course project",
        Path.cwd().parent.parent / "data" / "course project",
        Path("/content/ML_project/data/course project"),
        Path("/content/course project"),
    ]
    if drive.exists():
        candidates.extend([
            drive / "ML_project" / "data" / "course project",
            drive / "ML_project" / "course project",
            drive / "course project",
            drive / "新建文件夹" / "course project",
            drive / "新建文件夹" / "ML_project" / "data" / "course project",
        ])
        for child in drive.iterdir():
            if not child.is_dir() or child.name.startswith("."):
                continue
            candidates.append(child / "data" / "course project")
            candidates.append(child / "course project")
    # de-duplicate while preserving order
    seen, unique = set(), []
    for p in candidates:
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def resolve_data_root():
    override = DATA_ROOT_OVERRIDE
    if override not in (None, ""):
        p = Path(override)
        if _is_valid_data_root(p):
            return p
        raise FileNotFoundError(
            f"DATA_ROOT_OVERRIDE is set but not a valid dataset root: {p}\n"
            f"Expected subfolders like BCIC2A/train.h5 under this path."
        )

    env_root = os.environ.get("EEG_DATA_ROOT")
    if env_root:
        p = Path(env_root)
        if _is_valid_data_root(p):
            return p

    tried = []
    for p in _data_root_candidates():
        tried.append(str(p))
        if _is_valid_data_root(p):
            return p

    raise FileNotFoundError(
        "Cannot locate dataset root (need e.g. BCIC2A/train.h5).\n"
        "Tried:\n  - " + "\n  - ".join(tried) + "\n\n"
        "Colab fix: upload/sync the repo so Drive contains\n"
        "  MyDrive/ML_project/data/course project/{BCIC2A,...}/train.h5\n"
        "then set DATA_ROOT_OVERRIDE in the config cell, or clone to /content/ML_project."
    )


DATA_ROOT = resolve_data_root()
_DRIVE_DATA_SOURCE = DATA_ROOT if "drive/MyDrive" in str(DATA_ROOT) else None

# ========== 结构化输出目录 ==========
# 本地用 results/，Colab 用 _results/（临时），之后回写 Drive
RESULTS_DIR = Path.cwd().parent / "results"
if not RESULTS_DIR.exists():
    RESULTS_DIR = DATA_ROOT / "_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results dir: {RESULTS_DIR}")

# 记下 Drive 路径用于回写
DRIVE_ROOT = None
if "drive/MyDrive" in str(DATA_ROOT):
    DRIVE_ROOT = DATA_ROOT
    print(f"Drive 数据根目录: {DRIVE_ROOT}")

print(f"Data root: {DATA_ROOT}")

# 复制到本地缓存加速（Colab 下）
if USE_LOCAL_CACHE and Path("/content").exists():
    import shutil
    for DATA_NAME in DATASETS:
        src_base = DATA_ROOT / DATA_NAME
        dst_base = Path("/content/course project") / DATA_NAME
        dst_base.mkdir(parents=True, exist_ok=True)
        for fn in ["train.h5", "val.h5", "test_x_only.h5", "dataset_info.json", "dataset_info_fixed.json"]:
            src = src_base / fn
            dst = dst_base / fn
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
                print(f"  Copied {DATA_NAME}/{fn}")
    DATA_ROOT = Path("/content/course project")
    print(f"Using local cache at: {DATA_ROOT}")

# 反向探测 Drive 路径（本地缓存后 DATA_ROOT 已变，用之前记录或再搜一遍）
if DRIVE_ROOT is None and _DRIVE_DATA_SOURCE is not None:
    DRIVE_ROOT = _DRIVE_DATA_SOURCE
    print(f"Drive 数据根目录（来自缓存前）: {DRIVE_ROOT}")
elif DRIVE_ROOT is None and Path("/content/drive/MyDrive").exists():
    for candidate in _data_root_candidates():
        if "drive/MyDrive" in str(candidate) and _is_valid_data_root(candidate):
            DRIVE_ROOT = candidate
            print(f"自动检测到 Drive 路径: {DRIVE_ROOT}")
            break


FileNotFoundError: Cannot locate dataset root.

## 导入依赖 & 定义模型
包含 4 个模型定义：SimpleLinear、SimpleMLP、EEGNet、EEGGRU。

In [ ]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import sys
import time

# ---------- Dataset ----------
class TrainDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        with h5py.File(self.h5_path, "r") as f:
            self.x = torch.tensor(f["X"][()], dtype=torch.float32)
            self.y = torch.tensor(f["y"][()], dtype=torch.long)
        assert len(self.x) == len(self.y), "X and y length mismatch"

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


class TestDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        with h5py.File(self.h5_path, "r") as f:
            self.x = torch.tensor(f["X"][()], dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx]


# ---------- SimpleLinear ----------
class SimpleLinear(nn.Module):
    def __init__(self, input_channels, time_points, num_classes):
        super(SimpleLinear, self).__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(input_channels * time_points, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        return self.fc(x)


# ---------- SimpleMLP ----------
class SimpleMLP(nn.Module):
    def __init__(self, input_channels, num_classes=2, time_points=200,
                 hidden_dims=(256, 128), dropout=0.3):
        super().__init__()
        input_dim = input_channels * time_points
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h), nn.ReLU(), nn.Dropout(dropout)])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, num_classes))
        self.flatten = nn.Flatten()
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        x = self.flatten(x)
        logits = self.mlp(x)
        return logits


# ---------- EEGNet ----------
class EEGNet(nn.Module):
    def __init__(self, chans, time_point, num_classes,
                 f1=8, d=2, pk1=4, pk2=8, dp=0.5, max_norm1=1,
                 norm=torch.nn.Identity()):
        super().__init__()
        f2 = f1 * d
        self.block1 = nn.Sequential(
            nn.Conv2d(1, f1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(f1),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(f1, d * f1, (chans, 1), groups=f1, bias=False),
            nn.BatchNorm2d(d * f1),
            nn.ELU(),
            nn.AvgPool2d((1, pk1), stride=pk1),
            nn.Dropout(dp),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(d * f1, f2, (1, 16), groups=f2, bias=False, padding=(0, 8)),
            nn.Conv2d(f2, f2, kernel_size=1, bias=False),
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d((1, pk2), stride=pk2),
            nn.Dropout(dp),
        )
        self._apply_max_norm(self.block2[0], max_norm1)
        self.embed_dim = f2 * ((time_point // pk1) // pk2)
        self.classifier = nn.Linear(self.embed_dim, num_classes)
        self.norm = norm

    def _apply_max_norm(self, layer, max_norm):
        for name, param in layer.named_parameters():
            if "weight" in name:
                param.data = torch.renorm(param.data, p=2, dim=0, maxnorm=max_norm)

    def forward(self, x):
        x = self.norm(x)
        x = self.block1(x.unsqueeze(dim=1))
        x = self.block2(x)
        x = self.block3(x)
        feat = x.flatten(start_dim=1)
        logits = self.classifier(feat)
        return logits


# ---------- EEGGRU (from RNN.py) ----------
search_dirs = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "src",
    Path("/content"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/新建文件夹"),
    Path("/content/drive/MyDrive/新建文件夹/course project"),
]
if DATA_ROOT_OVERRIDE:
    search_dirs.insert(0, Path(DATA_ROOT_OVERRIDE).parent)

rnn_path = None
for d in search_dirs:
    p = d / "RNN.py"
    if p.exists():
        rnn_path = p
        break

if rnn_path is None:
    print("WARNING: RNN.py not found. EEGGRU model will be skipped.")
    EEGGRU = None
else:
    rnn_dir = str(rnn_path.parent)
    if rnn_dir not in sys.path:
        sys.path.append(rnn_dir)
    from RNN import EEGGRU
    print(f"Loaded EEGGRU from: {rnn_path}")

print("\nAll models defined. Ready to train!")

## 批量训练：所有数据集 × 所有模型
循环遍历每个数据集和每个模型，训练并记录结果。

In [ ]:
# ================================================================
# 训练入口 + 结构化结果保存
# ================================================================

results_summary = []
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

for DATA_NAME in DATASETS:
    base_dir = DATA_ROOT / DATA_NAME

    # 加载数据集信息
    info_path = base_dir / "dataset_info.json"
    info_path2 = base_dir / "dataset_info_fixed.json"
    chosen = info_path if info_path.exists() else info_path2
    if not chosen.exists():
        print(f"[SKIP] {DATA_NAME}: dataset_info.json not found")
        continue

    with open(chosen) as f:
        info = json.load(f)

    category_list = info["dataset"]["category_list"]
    num_classes = len(category_list)
    channels = info["dataset"]["channels"]
    target_sampling_rate = info["processing"]["target_sampling_rate"]
    window_sec = info["processing"]["window_sec"]

    print(f"\n{'='*60}")
    print(f"数据集: {DATA_NAME}")
    print(f"  类别 ({num_classes}): {category_list}")
    print(f"  通道数: {len(channels)}")
    print(f"  采样率: {target_sampling_rate} Hz, 窗口: {window_sec}s")
    print(f"{'='*60}")

    # 加载数据
    train_path = base_dir / "train.h5"
    val_path = base_dir / "val.h5"
    test_path = base_dir / "test_x_only.h5"

    for fname, p in [("train", train_path), ("val", val_path), ("test", test_path)]:
        if not p.exists():
            print(f"  [ERROR] {fname}.h5 not found at {p}")

    if not train_path.exists() or not test_path.exists():
        continue

    train_ds = TrainDataset(str(train_path))
    val_ds = TrainDataset(str(val_path)) if val_path.exists() else None
    test_ds = TestDataset(str(test_path))

    sample_x, _ = train_ds[0]
    CHANNELS, TIME_POINTS = sample_x.shape

    print(f"  数据形状: (N, C={CHANNELS}, T={TIME_POINTS})")
    print(f"  训练集: {len(train_ds)}, 验证集: {len(val_ds) if val_ds else 0}, 测试集: {len(test_ds)}")

    for MODEL_NAME in MODELS:
        # 跳过 EEGGRU 如果没找到 RNN.py
        if MODEL_NAME == "EEGGRU" and EEGGRU is None:
            print(f"\n  [SKIP] {MODEL_NAME}: RNN.py not found")
            continue

        # 检查是否已存在预测文件
        out_filename = f"{DATA_NAME}_{MODEL_NAME}.txt"
        out_path = base_dir / out_filename

        if SKIP_EXISTING and out_path.exists():
            print(f"\n  [SKIP] {MODEL_NAME}: {out_filename} already exists")
            results_summary.append({
                "dataset": DATA_NAME, "model": MODEL_NAME,
                "val_acc": None, "best_val_acc": None, "status": "skipped"
            })
            continue

        print(f"\n  --- 模型: {MODEL_NAME} ---")

        # ---------- 构建模型 ----------
        if MODEL_NAME == "SimpleLinear":
            model = SimpleLinear(CHANNELS, TIME_POINTS, num_classes)
        elif MODEL_NAME == "SimpleMLP":
            model = SimpleMLP(CHANNELS, num_classes, TIME_POINTS)
        elif MODEL_NAME == "EEGNet":
            model = EEGNet(CHANNELS, TIME_POINTS, num_classes)
        elif MODEL_NAME == "EEGGRU":
            model = EEGGRU(chans=CHANNELS, num_classes=num_classes)
        else:
            print(f"  Unknown model {MODEL_NAME}")
            continue

        model = model.to(device)

        # ---------- DataLoader ----------
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        if val_ds:
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

        # ---------- 损失 & 优化器 ----------
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        # ============================================================
        # 创建结构化输出目录: results/{DATA_NAME}/{MODEL_NAME}/{实验名}/
        # ============================================================
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        exp_parent = RESULTS_DIR / DATA_NAME / MODEL_NAME
        exp_parent.mkdir(parents=True, exist_ok=True)
        existing = sorted([d.name for d in exp_parent.iterdir() if d.is_dir()])
        next_num = 1
        if existing:
            nums = []
            for d in existing:
                try:
                    nums.append(int(d.split('_')[0]))
                except (ValueError, IndexError):
                    pass
            if nums:
                next_num = max(nums) + 1

        EXP_NAME = f"{next_num:03d}_{timestamp}_bs{BATCH_SIZE}_lr{LR}"
        EXP_DIR = exp_parent / EXP_NAME
        EXP_DIR.mkdir(parents=True, exist_ok=True)
        print(f"  实验目录: {EXP_DIR}")

        # 各结果文件路径
        CONFIG_PATH = EXP_DIR / "config.json"
        METRICS_PATH = EXP_DIR / "metrics.csv"
        LOSS_CURVE_PATH = EXP_DIR / "loss_curve.png"
        BEST_MODEL_PATH = EXP_DIR / "best_model.pt"
        PREDICTIONS_PATH = EXP_DIR / "predictions.txt"
        VAL_CM_PATH = EXP_DIR / "val_confusion_matrix.png"
        CLASS_REPORT_PATH = EXP_DIR / "classification_report.csv"
        FAILURE_CASES_PATH = EXP_DIR / "failure_cases.csv"

        # 保存实验配置
        config = {
            "dataset": DATA_NAME,
            "model": MODEL_NAME,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LR,
            "optimizer": "Adam",
            "loss_fn": "CrossEntropyLoss",
            "num_classes": num_classes,
            "channels": CHANNELS,
            "time_points": TIME_POINTS,
            "experiment_time": timestamp,
            "categories": category_list,
        }
        with open(CONFIG_PATH, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2, ensure_ascii=False)
        print(f"  Config saved")

        # ---------- 训练循环 ----------
        train_losses, val_losses, val_accs = [], [], []
        best_val_acc = 0.0
        t0 = time.time()

        for epoch in range(EPOCHS):
            # ===== Train =====
            model.train()
            train_loss_sum, train_num = 0.0, 0
            for data, label in train_loader:
                data, label = data.to(device), label.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, label)
                loss.backward()
                optimizer.step()
                train_loss_sum += loss.item() * label.size(0)
                train_num += label.size(0)
            epoch_train_loss = train_loss_sum / train_num
            train_losses.append(epoch_train_loss)

            # ===== Val =====
            if val_ds:
                model.eval()
                val_loss_sum, val_correct, val_num = 0.0, 0, 0
                with torch.no_grad():
                    for val_data, val_label in val_loader:
                        val_data, val_label = val_data.to(device), val_label.to(device)
                        val_output = model(val_data)
                        val_loss = criterion(val_output, val_label)
                        val_loss_sum += val_loss.item() * val_label.size(0)
                        val_num += val_label.size(0)
                        val_pred = torch.argmax(val_output, dim=1)
                        val_correct += (val_pred == val_label).sum().item()
                epoch_val_loss = val_loss_sum / val_num
                epoch_val_acc = val_correct / val_num
                val_losses.append(epoch_val_loss)
                val_accs.append(epoch_val_acc)

                # 保存 best checkpoint
                if epoch_val_acc > best_val_acc:
                    best_val_acc = epoch_val_acc
                    torch.save({
                        'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_acc': epoch_val_acc,
                        'val_loss': epoch_val_loss,
                        'config': config,
                    }, BEST_MODEL_PATH)

                if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
                    print(f"    Epoch [{epoch+1:02d}/{EPOCHS}] | "
                          f"Train Loss: {epoch_train_loss:.4f} | "
                          f"Val Loss: {epoch_val_loss:.4f} | "
                          f"Val Acc: {epoch_val_acc:.4f}")
            else:
                if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
                    print(f"    Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {epoch_train_loss:.4f}")

        elapsed = time.time() - t0
        final_val_acc = val_accs[-1] if val_accs else None

        print(f"  ✔ {MODEL_NAME} done | Time: {elapsed:.1f}s | "
              f"Best Val Acc: {best_val_acc:.4f} | Final Val Acc: {final_val_acc}")

        # ============================================================
        # 保存 metrics.csv
        # ============================================================
        if val_losses:
            metrics_df = pd.DataFrame({
                "epoch": list(range(1, len(train_losses) + 1)),
                "train_loss": train_losses,
                "val_loss": val_losses,
                "val_accuracy": val_accs,
            })
            metrics_df.to_csv(METRICS_PATH, index=False)
            print(f"  Metrics saved")

        # ============================================================
        # 保存 loss_curve.png
        # ============================================================
        if val_losses:
            import matplotlib
            matplotlib.use('Agg')
            import matplotlib.pyplot as plt
            plt.figure(figsize=(10, 6))
            plt.plot(range(1, EPOCHS + 1), train_losses, 'o-', label="Train Loss", linewidth=2, markersize=4)
            plt.plot(range(1, EPOCHS + 1), val_losses, 's-', label="Val Loss", linewidth=2, markersize=4)
            plt.xlabel("Epoch", fontsize=12)
            plt.ylabel("Loss", fontsize=12)
            plt.title(f"{DATA_NAME} / {MODEL_NAME} — Loss Curve", fontsize=14)
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)
            # 标注最佳 epoch 点
            best_ep = val_accs.index(best_val_acc) + 1 if best_val_acc > 0 else 0
            if best_ep > 0:
                plt.axvline(x=best_ep, color='gray', linestyle='--', alpha=0.5)
            plt.tight_layout()
            plt.savefig(str(LOSS_CURVE_PATH), dpi=150, bbox_inches='tight')
            plt.close()
            print(f"  Loss curve saved")

        # ============================================================
        # 验证集详细评估：混淆矩阵 + 分类报告 + failure cases
        # ============================================================
        if val_ds and val_losses:
            model.eval()
            all_val_preds, all_val_labels = [], []
            with torch.no_grad():
                for val_data, val_label in val_loader:
                    val_data = val_data.to(device)
                    val_output = model(val_data)
                    val_pred = torch.argmax(val_output, dim=1)
                    all_val_preds.extend(val_pred.cpu().tolist())
                    all_val_labels.extend(val_label.cpu().tolist())

            # 混淆矩阵
            cm = confusion_matrix(all_val_labels, all_val_preds)
            plt.figure(figsize=(8, 6))
            plt.imshow(cm, interpolation='nearest', cmap='Blues')
            plt.title(f"{DATA_NAME} / {MODEL_NAME} — Confusion Matrix", fontsize=14)
            plt.colorbar(shrink=0.8)
            tick_marks = np.arange(len(category_list))
            plt.xticks(tick_marks, category_list, rotation=45, ha='right')
            plt.yticks(tick_marks, category_list)
            plt.xlabel("Predicted", fontsize=12)
            plt.ylabel("True", fontsize=12)
            thresh = cm.max() / 2.0
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    plt.text(j, i, format(cm[i, j], 'd'),
                             ha="center", va="center",
                             color="white" if cm[i, j] > thresh else "black")
            plt.tight_layout()
            plt.savefig(str(VAL_CM_PATH), dpi=150, bbox_inches='tight')
            plt.close()
            print(f"  Confusion matrix saved")

            # 分类报告
            report_dict = classification_report(all_val_labels, all_val_preds,
                                                 target_names=category_list,
                                                 output_dict=True, zero_division=0)
            report_df = pd.DataFrame(report_dict).transpose()
            report_df.to_csv(CLASS_REPORT_PATH, float_format='%.4f')
            print(f"  Classification report saved")

            # Failure cases
            all_val_preds_arr = np.array(all_val_preds)
            all_val_labels_arr = np.array(all_val_labels)
            failure_mask = all_val_preds_arr != all_val_labels_arr
            if failure_mask.sum() > 0:
                failure_indices = np.where(failure_mask)[0]
                failure_df = pd.DataFrame({
                    "sample_index": failure_indices,
                    "true_label": all_val_labels_arr[failure_indices],
                    "true_category": [category_list[l] for l in all_val_labels_arr[failure_indices]],
                    "predicted_label": all_val_preds_arr[failure_indices],
                    "predicted_category": [category_list[l] for l in all_val_preds_arr[failure_indices]],
                })
                failure_df.to_csv(FAILURE_CASES_PATH, index=False)
                print(f"  Failure cases ({len(failure_df)} samples) saved")
            else:
                print(f"  No failure cases on validation set!")

        # ============================================================
        # 保存 test 预测标签（结构化目录 + 旧路径兼容）
        # ============================================================
        model.eval()
        # 用最佳模型权重推理
        if BEST_MODEL_PATH.exists():
            checkpoint = torch.load(BEST_MODEL_PATH, map_location='cpu')
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"  Loaded best model (epoch {checkpoint['epoch']}) for test inference")
        model = model.to(device)

        all_preds = []
        with torch.no_grad():
            for test_data in test_loader:
                test_data = test_data.to(device)
                test_output = model(test_data)
                test_pred = torch.argmax(test_output, dim=1)
                all_preds.extend(test_pred.cpu().tolist())

        # 结构化路径
        with open(PREDICTIONS_PATH, "w") as f:
            for label in all_preds:
                f.write(f"{int(label)}\n")
        print(f"  Saved {len(all_preds)} predictions to {PREDICTIONS_PATH.name}")

        # 旧路径兼容（数据集目录 + results/predictions/）
        with open(out_path, "w") as f:
            for label in all_preds:
                f.write(f"{int(label)}\n")
        unified_path = RESULTS_DIR / "predictions" / out_filename
        unified_path.parent.mkdir(parents=True, exist_ok=True)
        with open(unified_path, "w") as f:
            for label in all_preds:
                f.write(f"{int(label)}\n")

        # ============================================================
        # 追加到 experiments_summary.csv
        # ============================================================
        summary_path = RESULTS_DIR / "experiments_summary.csv"
        best_epoch_val = val_accs.index(best_val_acc) + 1 if val_accs and best_val_acc > 0 else 0
        n_params = sum(p.numel() for p in model.parameters())

        summary_entry = {
            "experiment_name": EXP_NAME,
            "dataset": DATA_NAME,
            "model": MODEL_NAME,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LR,
            "best_val_acc": f"{best_val_acc:.4f}",
            "final_val_acc": f"{final_val_acc:.4f}" if final_val_acc else "N/A",
            "best_epoch": best_epoch_val,
            "val_loss_final": f"{val_losses[-1]:.4f}" if val_losses else "N/A",
            "num_params": n_params,
            "time_sec": round(elapsed, 1),
            "experiment_time": timestamp,
            "experiment_dir": str(EXP_DIR),
            "status": "done",
        }
        entry_df = pd.DataFrame([summary_entry])

        if summary_path.exists():
            existing_df = pd.read_csv(summary_path)
            combined = pd.concat([existing_df, entry_df], ignore_index=True)
            combined.to_csv(summary_path, index=False)
        else:
            entry_df.to_csv(summary_path, index=False)

        # 也记录到内存
        results_summary.append({
            "dataset": DATA_NAME, "model": MODEL_NAME,
            "val_acc": round(final_val_acc, 4) if final_val_acc else None,
            "best_val_acc": round(best_val_acc, 4),
            "train_loss_final": round(train_losses[-1], 4),
            "epochs": EPOCHS, "time_sec": round(elapsed, 1),
            "status": "done",
            "experiment_dir": str(EXP_DIR),
        })

print(f"\n\n{'='*60}")
print("所有实验完成！")
print(f"{'='*60}")


## 实验结果汇总
展示所有数据集 × 模型的验证准确率。

In [ ]:
# ================================================================
# 实验结果汇总表（从 CSV 读取，兼容内存列表）
# ================================================================

# 优先从 CSV 读取（包含所有历史实验）
summary_path = RESULTS_DIR / "experiments_summary.csv"
if summary_path.exists():
    import pandas as pd
    df = pd.read_csv(summary_path)
    print(f"从 experiments_summary.csv 读取 {len(df)} 条实验记录\n")
else:
    # 从内存 results_summary 构建
    df = pd.DataFrame(results_summary)
    print(f"从内存读取 {len(df)} 条实验记录\n")

# 显示汇总表
if len(df) > 0:
    # 打印表格
    cols = ['dataset', 'model', 'best_val_acc', 'val_acc', 'time_sec', 'status']
    display_cols = [c for c in cols if c in df.columns]

    print(f"{'数据集':<12} {'模型':<16} {'Best Acc':<10} {'Val Acc':<10} {'Time':<8} {'状态':<10}")
    print("-" * 66)

    by_dataset = df.groupby('dataset') if 'dataset' in df.columns else [(None, df)]
    for dset, group in by_dataset:
        for i, (_, r) in enumerate(group.iterrows()):
            dname = dset if i == 0 else ""
            best = f"{r['best_val_acc']:.4f}" if pd.notna(r.get('best_val_acc')) and r.get('best_val_acc') != 'N/A' else "N/A"
            val = f"{r['val_acc']:.4f}" if pd.notna(r.get('val_acc')) and r.get('val_acc') != 'N/A' else "N/A"
            t = f"{r['time_sec']:.0f}s" if pd.notna(r.get('time_sec')) else "N/A"
            status = r.get('status', '')
            print(f"{dname:<12} {r['model']:<16} {best:<10} {val:<10} {t:<8} {status:<10}")

    print("-" * 66)

    # 最佳结果
    valid = df[pd.to_numeric(df['best_val_acc'], errors='coerce').notna()] if 'best_val_acc' in df.columns else df
    if len(valid) > 0:
        best_row = valid.loc[valid['best_val_acc'].astype(float).idxmax()]
        print(f"\n🏆 最佳结果: {best_row['dataset']} / {best_row['model']} = {float(best_row['best_val_acc']):.4f}")
    else:
        print("\n没有有效的实验结果")
else:
    print("没有实验记录")

# 保存内存中汇总到 JSON（兼容旧版回写）
with open(RESULTS_DIR / "summary.json", "w") as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)
print(f"\n内存汇总已保存到: {RESULTS_DIR / 'summary.json'}")


## 将结果回写到 Google Drive（可选）
如果前面启用了 `SYNC_TO_DRIVE`，这里会把 `_results/` 目录 + 各数据集的预测文件一起复制回 Drive。

In [ ]:
# ================================================================
# 将全部结果回写到 Google Drive（保持结构化目录）
# ================================================================

if SYNC_TO_DRIVE and DRIVE_ROOT is not None:
    import shutil
    from pathlib import Path

    print(f"\n正在回写结果到 Google Drive ...")
    print(f"  目标: {DRIVE_ROOT}")

    # 同步整个 _results/ 目录树到 Drive
    src_root = RESULTS_DIR
    dst_root = DRIVE_ROOT / "_results"

    # 用 shutil.copytree 保留完整目录结构
    if src_root.exists():
        dst_root.mkdir(parents=True, exist_ok=True)
        for item in src_root.rglob("*"):
            if item.is_file():
                rel_path = item.relative_to(src_root)
                dst_file = dst_root / rel_path
                dst_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(item, dst_file)
        print(f"  ✅ 完整结构化目录已回写:")
        print(f"     源: {src_root}")
        print(f"     目标: {dst_root}")

        # 列出回写文件摘要
        txt_files = list(dst_root.rglob("*.txt"))
        csv_files = list(dst_root.rglob("*.csv")) 
        pt_files = list(dst_root.rglob("*.pt"))
        png_files = list(dst_root.rglob("*.png"))
        json_files = list(dst_root.rglob("*.json"))
        print(f"     共: {len(txt_files)} 预测文件, {len(csv_files)} CSV, "
              f"{len(pt_files)} 模型, {len(png_files)} 图片, {len(json_files)} 配置")
    else:
        print(f"  ⚠️ 源目录不存在: {src_root}")

    # 也复制 predictions 到各数据集目录下（兼容旧版）
    pred_src = src_root / "predictions"
    if pred_src.exists():
        for f in pred_src.glob("*.txt"):
            # 从文件名推断数据集目录: SLEEP_EEGNet.txt → SLEEP/
            parts = f.stem.split("_", 1)
            if len(parts) == 2:
                dset_name = parts[0]
                dst_dir = DRIVE_ROOT / dset_name
                dst_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst_dir / f.name)

    # 同步 experiments_summary.csv 到 Drive 根目录
    summary_src = src_root / "experiments_summary.csv"
    if summary_src.exists():
        shutil.copy2(summary_src, DRIVE_ROOT / "experiments_summary.csv")
        print(f"  ✅ 汇总表已回写: experiments_summary.csv")

    print(f"\n✅ 所有结果已回写到: {DRIVE_ROOT}")
else:
    print("\n未启用 Drive 回写或未检测到 Drive 路径，跳过。")
    print(f"  SYNC_TO_DRIVE = {SYNC_TO_DRIVE}")
    print(f"  DRIVE_ROOT = {DRIVE_ROOT}")

print("\n完成！")


## 消融实验（Ablation Study）
选择一个数据集和一个模型，快速对比不同配置的影响。

In [ ]:
# ================================================================
# 消融实验配置
# ================================================================

ABLATION_DATASET = "SLEEP"   # 选一个数据集做消融
ABLATION_BASE_MODEL = "EEGNet"  # 基准模型

# 消融实验列表：每组修改 model_kwargs 中的一个参数
ablation_configs = [
    {"name": "Baseline",       "model_kwargs": {}},
    {"name": "No Dropout",     "model_kwargs": {"dp": 0.0}},
    {"name": "F1=4",           "model_kwargs": {"f1": 4}},
    {"name": "F1=16",          "model_kwargs": {"f1": 16}},
    {"name": "Pool=8",         "model_kwargs": {"pk1": 8}},
]

print(f"消融实验: {ABLATION_BASE_MODEL} @ {ABLATION_DATASET}")
for cfg in ablation_configs:
    print(f"  {cfg['name']}: {cfg['model_kwargs']}")

# 运行消融实验
base_dir = DATA_ROOT / ABLATION_DATASET
info_path = base_dir / "dataset_info.json"
if not info_path.exists():
    info_path = base_dir / "dataset_info_fixed.json"

if not info_path.exists():
    print("数据集信息文件不存在，跳过消融实验。")
else:
    with open(info_path) as f:
        info = json.load(f)
    num_classes = len(info["dataset"]["category_list"])

    train_ds = TrainDataset(str(base_dir / "train.h5"))
    val_ds = TrainDataset(str(base_dir / "val.h5"))
    sample_x, _ = train_ds[0]
    CH, TP = sample_x.shape

    print(f"\n{'配置':<16} {'Val Acc':<10} {'Best Acc':<10}")
    print("-" * 40)

    ablation_results = []

    for cfg in ablation_configs:
        kwargs = {"chans": CH, "time_point": TP, "num_classes": num_classes}
        kwargs.update(cfg["model_kwargs"])

        model = EEGNet(**kwargs).to(device)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_acc = 0.0
        for epoch in range(EPOCHS):
            model.train()
            for data, label in train_loader:
                data, label = data.to(device), label.to(device)
                optimizer.zero_grad()
                loss = criterion(model(data), label)
                loss.backward()
                optimizer.step()

            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for data, label in val_loader:
                    data, label = data.to(device), label.to(device)
                    pred = torch.argmax(model(data), dim=1)
                    correct += (pred == label).sum().item()
                    total += label.size(0)
            acc = correct / total
            best_acc = max(best_acc, acc)

        print(f"{cfg['name']:<16} {acc:<10.4f} {best_acc:<10.4f}")
        ablation_results.append({
            "name": cfg["name"],
            "kwargs": cfg["model_kwargs"],
            "final_val_acc": round(acc, 4),
            "best_val_acc": round(best_acc, 4),
        })

    # 保存消融实验结果
    ablation_path = RESULTS_DIR / f"ablation_{ABLATION_DATASET}_{ABLATION_BASE_MODEL}.json"
    # ablation_path.parent already exists
    with open(ablation_path, "w") as f:
        json.dump({
            "dataset": ABLATION_DATASET,
            "base_model": ABLATION_BASE_MODEL,
            "configs": ablation_results,
        }, f, indent=2, ensure_ascii=False)
    print(f"\n消融结果已保存到: {ablation_path}")

    # 如果启用了 Drive 回写，也复制过去
    if SYNC_TO_DRIVE and DRIVE_ROOT is not None:
        import shutil
        dst_ablation = DRIVE_ROOT / "_results" / ablation_path.name
        dst_ablation.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ablation_path, dst_ablation)
        print(f"  \u2714 消融结果已回写到 Drive: {dst_ablation}")

print("\n消融实验完成！")

## 绘制训练曲线（选做）
如果想看某个实验的 Loss 曲线，可以用下面的代码单独跑。

In [ ]:
# 可选：绘制某个实验的 loss 曲线
# 需要在训练时保存了 train_losses / val_losses / val_accs 变量

if 'train_losses' in dir() and train_losses:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label="Train Loss")
    if 'val_losses' in dir() and val_losses:
        plt.plot(val_losses, label="Val Loss")
        plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curve")

    if 'val_accs' in dir() and val_accs:
        plt.subplot(1, 2, 2)
        plt.plot(val_accs, label="Val Acc", color="green")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title("Validation Accuracy")
        plt.legend()

    plt.tight_layout()
    # 保存图像
    plot_path = RESULTS_DIR / "loss_curve.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    print(f"Loss 曲线已保存到: {plot_path}")

    # 如果启用了 Drive 回写，也复制过去
    if SYNC_TO_DRIVE and DRIVE_ROOT is not None:
        import shutil
        dst_plot = DRIVE_ROOT / "_results" / "loss_curve.png"
        dst_plot.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(plot_path, dst_plot)
        print(f"  \u2714 Loss 曲线已回写到 Drive: {dst_plot}")

    plt.show()
else:
    print("没有训练数据可绘制。请先在训练 Cell 中运行后再执行本 Cell。")